# 🛠️ Notebook 2: Amazon Shopping — Implementation

We'll build the store in **three passes**:
1. **v1 (bad):** works on the happy path but has real bugs.
2. **v2 (better):** fix the bugs — freeze prices, proper `Order` object, Strategy for payment.
3. **v3 (good):** add **Observer** (notifications), **State** transitions, discounts, shipping & tax.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/amazon-shopping
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## v1 — ❌ First attempt (buggy, on purpose)

- Order holds a **reference** to the live cart, and the total is computed on demand from the **current** catalog price.
- Payment is hard-coded (no Strategy).
- Stock check and deduction happen in two steps with a gap — bugs hide there.


In [ ]:
# v1 -- intentionally broken design
from dataclasses import dataclass

@dataclass
class ProductV1:
    sku: str; name: str; price: float; stock: int

class CartV1:
    def __init__(self):
        self.lines = {}  # sku -> qty
    def add(self, p, qty):
        self.lines[p.sku] = self.lines.get(p.sku, 0) + qty

class StoreV1:
    def __init__(self, products):
        self.catalog = {p.sku: p for p in products}
    def checkout(self, cart, card_number):
        # BUG 1: no stock check -> stock can go negative
        for sku, qty in cart.lines.items():
            self.catalog[sku].stock -= qty
        # BUG 2: "order" just references the live cart.lines
        order = {"items": cart.lines, "card": card_number}
        # BUG 3: total computed on demand from current prices
        order["total"] = lambda: sum(self.catalog[s].price * q for s, q in order["items"].items())
        print("charged to card ****" + card_number[-4:])
        return order

# demo the bugs
store = StoreV1([ProductV1("BOOK-1", "Clean Code", 25.0, 1)])
cart  = CartV1()
cart.add(store.catalog["BOOK-1"], 2)   # asks for 2 but only 1 in stock!
order = store.checkout(cart, "4111111111111111")

print("stock now:", store.catalog["BOOK-1"].stock, " (oops, negative -- we oversold)")
print("order total right now: $", order["total"]())

# Price changes later -- the old order total silently changes too!
store.catalog["BOOK-1"].price = 1.0
print("order total after price change: $", order["total"]())

# Also: cart and order share the same dict, so editing cart mutates the order.
cart.lines["BOOK-1"] = 99
print("order items after cart edit:", order["items"])


### Bugs we just demonstrated

1. **Oversell** — no stock validation.
2. **Order mutates with the catalog** — changing a product's price retroactively rewrites paid orders.
3. **Order mutates with the cart** — the order and cart share the same `dict`.
4. **Payment is hard-coded** to credit-card numbers.

Let's fix them.


## v2 — ✅ Freeze the order, add Payment **Strategy**

Key ideas:
- `Order` stores a **copy** of the cart lines and the total at checkout time.
- Stock is checked **then** deducted (fail fast if something's out of stock).
- `PaymentMethod` is an interface; concrete payments are plug-ins.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
import itertools

class OrderStatus(Enum):
    PENDING = 1; PAID = 2; SHIPPED = 3; CANCELLED = 4

@dataclass
class Product:
    sku: str; name: str; price: float; stock: int

class Cart:
    def __init__(self):
        self.lines: dict[str, int] = {}   # sku -> qty
    def add(self, product: Product, qty: int):
        if qty <= 0:
            raise ValueError("qty must be positive")
        if qty > product.stock:
            raise RuntimeError(f"not enough stock for {product.sku}")
        self.lines[product.sku] = self.lines.get(product.sku, 0) + qty
    def remove(self, sku: str):
        self.lines.pop(sku, None)
    def total(self, catalog: dict) -> float:
        return sum(catalog[sku].price * q for sku, q in self.lines.items())

@dataclass
class Order:
    id: int
    lines: dict            # frozen copy of sku -> qty
    total: float           # frozen at checkout
    status: OrderStatus = OrderStatus.PENDING

# --- Strategy pattern: swap payment methods without touching the Store ---
class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount: float) -> bool: ...

class CreditCard(PaymentMethod):
    def __init__(self, number, cvv): self.number, self.cvv = number, cvv
    def pay(self, amount):
        print(f"  [card] charged ${amount:.2f} to ****{self.number[-4:]}")
        return True

class PayPal(PaymentMethod):
    def __init__(self, email): self.email = email
    def pay(self, amount):
        print(f"  [paypal] charged ${amount:.2f} to {self.email}")
        return True

class Store:
    _oid = itertools.count(1)
    def __init__(self, products):
        self.catalog = {p.sku: p for p in products}
        self.orders: list = []

    def checkout(self, cart: Cart, method: PaymentMethod) -> Order:
        # 1) validate stock up front (fail fast)
        for sku, qty in cart.lines.items():
            if self.catalog[sku].stock < qty:
                raise RuntimeError(f"{sku} out of stock")
        # 2) deduct stock
        for sku, qty in cart.lines.items():
            self.catalog[sku].stock -= qty
        # 3) snapshot the cart into an Order (prices frozen)
        order = Order(
            id=next(Store._oid),
            lines=dict(cart.lines),
            total=cart.total(self.catalog),
        )
        # 4) charge via Strategy
        if method.pay(order.total):
            order.status = OrderStatus.PAID
        self.orders.append(order)
        return order

# --- demo ---
store = Store([
    Product("BOOK-1", "Clean Code", 25.0, stock=3),
    Product("MUG-1",  "Coffee Mug", 10.0, stock=5),
])

cart = Cart()
cart.add(store.catalog["BOOK-1"], 2)
cart.add(store.catalog["MUG-1"], 1)
print("cart total:", cart.total(store.catalog))

order = store.checkout(cart, CreditCard("4111111111111111", "123"))
print("order", order.id, "status:", order.status.name, "total:", order.total)
print("book stock left:", store.catalog["BOOK-1"].stock)

# Prove the order is frozen: changing catalog price does NOT change the order total
store.catalog["BOOK-1"].price = 1.0
print("order total still:", order.total)


### What we fixed in v2

- ✅ **Price freeze** — the order stored `total` by value, not by function.
- ✅ **No oversell** — stock checked before deduction.
- ✅ **Open/Closed Principle** — adding `ApplePay` is a new `PaymentMethod` subclass. `Store` doesn't change.
- ✅ **Fail fast** — if any item is out of stock, we abort before touching anything.

### Testing the error path


In [ ]:
# Oversell now raises instead of silently breaking
tight_store = Store([Product("P", "Pen", 1.0, stock=1)])
c = Cart()
try:
    c.add(tight_store.catalog["P"], 5)
except RuntimeError as e:
    print("expected:", e)


## v3 — 🌟 Real-world extras: Observer, State, Discounts, Shipping & Tax

Real checkout flows also need:
- **Notifications** when order status changes (email the customer, tell the warehouse).
- **Legal state transitions** (you can't ship a cancelled order).
- **Discounts / coupons** — another Strategy.
- **Shipping + tax** — computed at checkout.

We'll extend (not replace) the v2 classes.


In [ ]:
# --- Observer pattern: listeners react to order status changes ---
class EmailNotifier:
    def __init__(self, to): self.to = to
    def on_status_change(self, order, old, new):
        print(f"  [email -> {self.to}] order #{order.id} {old.name} -> {new.name}")

class WarehouseNotifier:
    def on_status_change(self, order, old, new):
        if new is OrderStatus.PAID:
            print(f"  [warehouse] pick & pack order #{order.id}")

# --- State transitions enforced on Order ---
LEGAL_TRANSITIONS = {
    OrderStatus.PENDING:   {OrderStatus.PAID, OrderStatus.CANCELLED},
    OrderStatus.PAID:      {OrderStatus.SHIPPED, OrderStatus.CANCELLED},
    OrderStatus.SHIPPED:   set(),
    OrderStatus.CANCELLED: set(),
}

class ObservableOrder(Order):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self._observers = []
    def subscribe(self, obs):
        self._observers.append(obs)
    def set_status(self, new: OrderStatus):
        if new not in LEGAL_TRANSITIONS[self.status]:
            raise RuntimeError(f"illegal transition {self.status.name} -> {new.name}")
        old, self.status = self.status, new
        for o in self._observers:
            o.on_status_change(self, old, new)

# --- Discount Strategy ---
class Discount(ABC):
    @abstractmethod
    def apply(self, subtotal: float) -> float: ...

class NoDiscount(Discount):
    def apply(self, subtotal): return subtotal

class PercentOff(Discount):
    def __init__(self, percent): self.percent = percent
    def apply(self, subtotal): return subtotal * (1 - self.percent / 100)

class CouponCode(Discount):
    # SAVE10 -> $10 off on orders >= $50
    def __init__(self, code): self.code = code
    def apply(self, subtotal):
        if self.code == "SAVE10" and subtotal >= 50:
            return subtotal - 10
        return subtotal

# --- Shipping + Tax ---
def shipping_fee(subtotal: float) -> float:
    # flat $5, free when subtotal >= $50
    return 0.0 if subtotal >= 50 else 5.0

def tax(amount: float, rate: float = 0.08) -> float:
    return round(amount * rate, 2)

# --- Enhanced Store ---
class Store2(Store):
    def __init__(self, products, observers=None):
        super().__init__(products)
        self.default_observers = observers or []

    def checkout(self, cart: Cart, method: PaymentMethod,
                 discount: Discount = NoDiscount()) -> ObservableOrder:
        for sku, qty in cart.lines.items():
            if self.catalog[sku].stock < qty:
                raise RuntimeError(f"{sku} out of stock")
        for sku, qty in cart.lines.items():
            self.catalog[sku].stock -= qty

        subtotal   = cart.total(self.catalog)
        discounted = discount.apply(subtotal)
        ship       = shipping_fee(discounted)
        total      = round(discounted + ship + tax(discounted + ship), 2)

        order = ObservableOrder(
            id=next(Store._oid),
            lines=dict(cart.lines),
            total=total,
        )
        for o in self.default_observers:
            order.subscribe(o)

        if method.pay(order.total):
            order.set_status(OrderStatus.PAID)
        self.orders.append(order)
        return order

# --- demo ---
store = Store2(
    [Product("BOOK-1", "Clean Code", 25.0, 3),
     Product("MUG-1",  "Coffee Mug", 10.0, 5)],
    observers=[EmailNotifier("alice@example.com"), WarehouseNotifier()],
)

cart = Cart()
cart.add(store.catalog["BOOK-1"], 2)  # $50 -> free shipping, SAVE10 applies
cart.add(store.catalog["MUG-1"],  1)  # +$10

order = store.checkout(cart, PayPal("alice@example.com"), discount=CouponCode("SAVE10"))
print(f"final total (incl. tax): ${order.total}")

# Advance the order through its legal states
order.set_status(OrderStatus.SHIPPED)

# Illegal transition -> raises
try:
    order.set_status(OrderStatus.PAID)
except RuntimeError as e:
    print("expected:", e)


### What each pattern bought us

| Pattern   | Benefit in this lab                                                              |
|-----------|----------------------------------------------------------------------------------|
| Strategy  | Swap `PaymentMethod` and `Discount` without touching `Store.checkout`.           |
| Observer  | Add `SMSNotifier`, `SlackNotifier`, `AnalyticsTracker` — none know about each other. |
| State     | `ObservableOrder.set_status` prevents illegal moves like `CANCELLED -> PAID`.     |
| Snapshot  | `Order.lines` and `Order.total` are frozen — safe even if the catalog mutates.   |

### 🧪 Mini exercise — try these

1. Add an `ApplePay` payment method. Notice `Store2` does **not** change.
2. Add `BuyOneGetOneFree` discount for a specific SKU.
3. Add an `AnalyticsTracker` observer that counts how many orders reached `SHIPPED`.
4. Make `Store2.checkout` **thread-safe** with a `threading.Lock` around the stock check+deduct.

### ⚠️ Concurrency note — real stores need locks

In `Store2.checkout`, the check-then-deduct sequence is a **race condition** if two
shoppers call it at the same time:

```
Thread A: check (stock=1, ok)     Thread B: check (stock=1, ok)
Thread A: deduct (stock=0)        Thread B: deduct (stock=-1)  <- oversold
```

In a single process, fix with `threading.Lock`. In a distributed system, use
optimistic locking (a `version` column) or a transactional DB
`UPDATE ... SET stock = stock - :q WHERE sku = :s AND stock >= :q`.

### 👉 Next steps you could try
- Add **returns / refunds** (new state `REFUNDED`; refund via the same `PaymentMethod`).
- Add a **Repository** layer so orders/products come from a DB instead of memory.
- Add **inventory reservation** with a TTL (hold stock for 10 min while the user pays).
- Add **Prime** shipping tier with different rules.
